In [ ]:
import pennylane as qml
from pennylane import numpy as np
from collections import Counter

In [ ]:
from solvers import SolverFactory, DynamicSolver
import map
import pathFormulation
import config.parser as config_parser
import QUBOBuilder
from config.hdf5parser import load_map_from_hdf5

In [ ]:
conf = config_parser.load_config("config/config.yaml", sections=["problems", "penalty_sets", "solver"])
map_hdf5 = load_map_from_hdf5("maps/synthetic/2x2/no_obs2x2.h5")
grid = map.Grid.from_hdf5_data(map_hdf5)
map_conf = conf["problems"]["grid_2x2_no_obs"]
problem = pathFormulation.PathfindingProblem.from_hdf5_data(map_hdf5, map_conf)

In [ ]:
material_costs = {
    "ceramic": 1.0,
    "water": 5.0,
    "asphalt": 1.5,
    "grass": 2.0
}
# penalties_conf = conf["penalty_sets"]["ter"]
penalties_conf = conf["penalty_sets"]["alt_later"]
builder = QUBOBuilder.QUBOBuilder(problem, penalties=penalties_conf, name="standard", material_cost=material_costs)

In [ ]:
builder.build()

# Be mind that each params needs to be len p (layers)
# Note that there is little variation 0.1-.4
# init_params = np.array([[1.18846312, 0.80366844], [0.35609933, 0.59871356]], requires_grad=True)
init_params = np.array([[1.182179, 0.78571062], [0.36629231, 0.59672656]], requires_grad=True)

# Use default.qubit instead of lightning.qubit for testing
pennylane_solver = SolverFactory.create_solver(
        backend="pennylane", normalize_scale=2.0, num_reads=1000, 
        layers=2, optimizer="GradientDescent", opt_steps=30,
        device="default.qubit", params=init_params)

solution = pennylane_solver.solve_qubo(builder)
print("Solution completed successfully!")

In [ ]:
print("Solution:", solution)
print("Path:", pennylane_solver.decode_path(solution["solution"], problem))
print("Parameters", solution["optimized_params"])